# Hospital Revenue — Statistical Analysis

Notebook phân tích doanh thu bệnh viện theo 4 nhóm:
1. **Data Loading** — Load và chuẩn bị dữ liệu từ `hospital.db`
2. **Descriptive Statistics** — Phân phối, thống kê mô tả
3. **Correlation Analysis** — Tương quan giữa khoa/dịch vụ
4. **Trend Analysis** — Xu hướng, mùa vụ, tăng trưởng
5. **Export** — Xuất kết quả ra JSON cho dashboard

> Sau khi chạy xong cell cuối, dashboard sẽ tự đọc file JSON mới nhất.

## Section 1 — Data Loading & Preparation

In [1]:
import sqlite3
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DB_PATH = Path("../backend/hospital.db")
OUTPUT_DIR = Path("../backend/analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [2]:
conn = sqlite3.connect(DB_PATH)

sql = """
    SELECT
        f.ID, f.NGAY, f.ID_BENHNHAN, f.ID_KHAMBENH,
        f.SOLUONG, f.DONGIA, f.THANHTIEN, f.BHYTTRA, f.NGUOIBENHTRA,
        k.TEN_KHOAPHONG, k.LOAI_KHOA,
        d.TEN_DICHVU, d.LOAI_DICHVU, d.NHOM_DICHVU,
        dt.TEN_DOITUONG,
        ldt.TEN_LOAI_DIEUTRI
    FROM FACT_DOANHTHU f
    LEFT JOIN DIM_KHOAPHONG k   ON f.ID_KHOAPHONG   = k.ID_KHOAPHONG
    LEFT JOIN DIM_DICHVU d      ON f.ID_DICHVU       = d.ID_DICHVU
    LEFT JOIN DIM_DOITUONG dt   ON f.ID_DOITUONG     = dt.ID_DOITUONG
    LEFT JOIN DIM_LOAI_DIEUTRI ldt ON f.ID_LOAI_DIEUTRI = ldt.ID_LOAI_DIEUTRI
"""

df = pd.read_sql_query(sql, conn)
conn.close()

df["NGAY"] = pd.to_datetime(df["NGAY"])
df["THANG"]     = df["NGAY"].dt.month
df["NAM"]       = df["NGAY"].dt.year
df["QUY"]       = df["NGAY"].dt.quarter
df["THU"]       = df["NGAY"].dt.dayofweek   # 0=Mon
df["THU_TEN"]   = df["NGAY"].dt.day_name()
df["THANG_NAM"] = df["NGAY"].dt.to_period("M").astype(str)

print(f"Tổng số bản ghi : {len(df):,}")
print(f"Khoảng thời gian: {df['NGAY'].min().date()} → {df['NGAY'].max().date()}")
df.head()

Tổng số bản ghi : 7,233
Khoảng thời gian: 2026-01-01 → 2026-05-10


,ID,NGAY,ID_BENHNHAN,ID_KHAMBENH,SOLUONG,DONGIA,THANHTIEN,BHYTTRA,NGUOIBENHTRA,TEN_KHOAPHONG,...,LOAI_DICHVU,NHOM_DICHVU,TEN_DOITUONG,TEN_LOAI_DIEUTRI,THANG,NAM,QUY,THU,THU_TEN,THANG_NAM
0,1,2026-01-01,BN00139,KB202601019817,2,8334897.0,16669794.0,6667918.0,10001877.0,Khoa Sản,...,Phẫu thuật,Phẫu thuật / thủ thuật,BHYT,Nội trú,1,2026,1,3,Thursday,2026-01
1,2,2026-01-01,BN00045,KB202601015569,3,89527.0,268582.0,0.0,268582.0,Khoa Sản,...,Xét nghiệm,Xét nghiệm,Dịch vụ,Nội trú,1,2026,1,3,Thursday,2026-01
2,3,2026-01-01,BN00015,KB202601019098,2,118210.0,236420.0,189136.0,47284.0,Khoa Chẩn đoán hình ảnh,...,Xét nghiệm,Xét nghiệm,BHYT,Ngoại trú,1,2026,1,3,Thursday,2026-01
3,4,2026-01-01,BN00142,KB202601011166,2,93491.0,186983.0,74793.0,112190.0,Khoa Nhi,...,Xét nghiệm,Xét nghiệm,BHYT,Nội trú,1,2026,1,3,Thursday,2026-01
4,5,2026-01-01,BN00244,KB202601019041,3,144124.0,432372.0,345897.0,86474.0,Khoa Sản,...,Khám bệnh,Khám bệnh,BHYT,Ngoại trú,1,2026,1,3,Thursday,2026-01


## Section 2 — Descriptive Statistics

In [3]:
# ── Overall stats ────────────────────────────────────────────────────
overall_stats = {
    "mean"  : float(df["THANHTIEN"].mean()),
    "median": float(df["THANHTIEN"].median()),
    "std"   : float(df["THANHTIEN"].std()),
    "p25"   : float(df["THANHTIEN"].quantile(0.25)),
    "p75"   : float(df["THANHTIEN"].quantile(0.75)),
    "min"   : float(df["THANHTIEN"].min()),
    "max"   : float(df["THANHTIEN"].max()),
    "total" : float(df["THANHTIEN"].sum()),
    "count" : int(len(df)),
}

# ── BHYT ratio ───────────────────────────────────────────────────────
total_thanhtien = df["THANHTIEN"].sum()
total_bhyt      = df["BHYTTRA"].sum()
total_patient   = df["NGUOIBENHTRA"].sum()
bhyt_ratio      = total_bhyt / total_thanhtien * 100 if total_thanhtien > 0 else 0

unique_patients    = int(df["ID_BENHNHAN"].nunique())
unique_visits      = int(df["ID_KHAMBENH"].nunique())
revenue_per_patient = float(total_thanhtien / unique_patients) if unique_patients else 0
revenue_per_visit   = float(total_thanhtien / unique_visits)   if unique_visits   else 0

# ── By department ────────────────────────────────────────────────────
dept_stats = (
    df.groupby("TEN_KHOAPHONG")
    .agg(
        total           = ("THANHTIEN", "sum"),
        mean            = ("THANHTIEN", "mean"),
        median          = ("THANHTIEN", "median"),
        std             = ("THANHTIEN", "std"),
        count           = ("THANHTIEN", "count"),
        unique_patients = ("ID_BENHNHAN", "nunique"),
        unique_visits   = ("ID_KHAMBENH", "nunique"),
    )
    .reset_index()
)
dept_stats["revenue_per_visit"] = (
    dept_stats["total"] / dept_stats["unique_visits"].replace(0, np.nan)
).fillna(0)

# ── By service group ─────────────────────────────────────────────────
service_stats = (
    df.groupby("NHOM_DICHVU")
    .agg(total=("THANHTIEN", "sum"), mean=("THANHTIEN", "mean"), count=("THANHTIEN", "count"))
    .reset_index()
)

# ── By day of week ───────────────────────────────────────────────────
dow_stats = (
    df.groupby(["THU", "THU_TEN"])
    .agg(total=("THANHTIEN", "sum"), mean=("THANHTIEN", "mean"), count=("THANHTIEN", "count"))
    .reset_index()
    .sort_values("THU")
)

day_name_vi = {
    "Monday": "Thứ Hai", "Tuesday": "Thứ Ba", "Wednesday": "Thứ Tư",
    "Thursday": "Thứ Năm", "Friday": "Thứ Sáu", "Saturday": "Thứ Bảy", "Sunday": "Chủ Nhật"
}
dow_stats["THU_VI"] = dow_stats["THU_TEN"].map(day_name_vi)

print("✅ Thống kê mô tả đã tính xong.")
print(f"  BHYT ratio     : {bhyt_ratio:.1f}%")
print(f"  Revenue/patient: {revenue_per_patient:,.0f} VNĐ")
print(f"  Revenue/visit  : {revenue_per_visit:,.0f} VNĐ")

✅ Thống kê mô tả đã tính xong.
  BHYT ratio     : 35.2%
  Revenue/patient: 23,217,165 VNĐ
  Revenue/visit  : 2,304,313 VNĐ


In [4]:
# Histogram phân phối doanh thu
fig = px.histogram(
    df, x="THANHTIEN", nbins=60,
    title="Phân phối doanh thu theo giao dịch",
    labels={"THANHTIEN": "Thành tiền (VNĐ)", "count": "Số giao dịch"},
    color_discrete_sequence=["#1976d2"],
)
fig.add_vline(x=overall_stats["median"], line_dash="dash", line_color="#e53935",
              annotation_text=f"Median: {overall_stats['median']:,.0f}", annotation_position="top right")
fig.add_vline(x=overall_stats["mean"], line_dash="dot", line_color="#fb8c00",
              annotation_text=f"Mean: {overall_stats['mean']:,.0f}", annotation_position="top left")
fig.update_layout(bargap=0.05)
fig.show()

In [5]:
# Box plot theo khoa
dept_order = dept_stats.sort_values("median", ascending=False)["TEN_KHOAPHONG"].tolist()
fig = px.box(
    df, x="TEN_KHOAPHONG", y="THANHTIEN",
    category_orders={"TEN_KHOAPHONG": dept_order},
    title="Phân phối doanh thu theo khoa/phòng (sắp xếp theo median)",
    labels={"TEN_KHOAPHONG": "", "THANHTIEN": "Thành tiền (VNĐ)"},
    color="TEN_KHOAPHONG",
)
fig.update_layout(showlegend=False, xaxis_tickangle=-35)
fig.show()

In [6]:
# Cơ cấu BHYT vs tự trả
fig = go.Figure(go.Pie(
    labels=["BHYT trả", "Người bệnh trả"],
    values=[total_bhyt, total_patient],
    hole=0.45,
    marker_colors=["#1976d2", "#8e24aa"],
    textinfo="label+percent",
))
fig.update_layout(title="Cơ cấu nguồn thanh toán (toàn thời gian)")
fig.show()

In [7]:
# Doanh thu trung bình mỗi lượt khám theo khoa
fig = px.bar(
    dept_stats.sort_values("revenue_per_visit"),
    x="revenue_per_visit", y="TEN_KHOAPHONG",
    orientation="h",
    title="Doanh thu trung bình mỗi lượt khám theo khoa",
    labels={"revenue_per_visit": "Doanh thu/lượt (VNĐ)", "TEN_KHOAPHONG": ""},
    color="revenue_per_visit",
    color_continuous_scale="Blues",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [8]:
# Heatmap: ngày trong tuần × nhóm dịch vụ
dow_service = (
    df.groupby(["THU", "THU_TEN", "NHOM_DICHVU"])["THANHTIEN"]
    .mean().reset_index()
)
pivot = dow_service.pivot_table(
    index="THU_TEN", columns="NHOM_DICHVU", values="THANHTIEN", aggfunc="mean"
)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = pivot.reindex([d for d in day_order if d in pivot.index])
pivot.index = [day_name_vi.get(d, d) for d in pivot.index]

fig = px.imshow(
    pivot,
    title="Doanh thu TB theo ngày trong tuần × nhóm dịch vụ",
    labels={"color": "Doanh thu TB (VNĐ)"},
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=",.0f",
)
fig.show()

## Section 3 — Correlation Analysis

In [9]:
# ── Department daily pivot → correlation ─────────────────────────────
daily_dept = df.groupby(["NGAY", "TEN_KHOAPHONG"])["THANHTIEN"].sum().reset_index()
pivot_dept = daily_dept.pivot_table(
    index="NGAY", columns="TEN_KHOAPHONG", values="THANHTIEN", fill_value=0
)
corr_dept = pivot_dept.corr()

# ── Service group daily pivot → correlation ──────────────────────────
daily_svc = df.groupby(["NGAY", "NHOM_DICHVU"])["THANHTIEN"].sum().reset_index()
pivot_svc = daily_svc.pivot_table(
    index="NGAY", columns="NHOM_DICHVU", values="THANHTIEN", fill_value=0
)
corr_svc = pivot_svc.corr()

# ── Volume vs Revenue per department ─────────────────────────────────
dept_vol_rev = (
    df.groupby("TEN_KHOAPHONG")
    .agg(
        total_revenue   = ("THANHTIEN", "sum"),
        total_visits    = ("ID_KHAMBENH", "nunique"),
        total_patients  = ("ID_BENHNHAN", "nunique"),
    )
    .reset_index()
)

print("✅ Ma trận tương quan đã tính xong.")

✅ Ma trận tương quan đã tính xong.


In [10]:
# Heatmap tương quan giữa các khoa
fig = px.imshow(
    corr_dept,
    title="Ma trận tương quan doanh thu giữa các khoa/phòng (theo ngày)",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    text_auto=".2f",
    aspect="auto",
)
fig.show()

In [11]:
# Heatmap tương quan giữa các nhóm dịch vụ
fig = px.imshow(
    corr_svc,
    title="Ma trận tương quan doanh thu giữa các nhóm dịch vụ (theo ngày)",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    text_auto=".2f",
    aspect="auto",
)
fig.show()

In [12]:
# Scatter: số lượt khám vs doanh thu (bubble = số bệnh nhân)
fig = px.scatter(
    dept_vol_rev,
    x="total_visits", y="total_revenue",
    size="total_patients", color="TEN_KHOAPHONG",
    text="TEN_KHOAPHONG",
    title="Số lượt khám vs Doanh thu theo khoa (kích thước = số bệnh nhân)",
    labels={"total_visits": "Số lượt khám", "total_revenue": "Tổng doanh thu (VNĐ)"},
)
fig.update_traces(textposition="top center")
fig.update_layout(showlegend=False)
fig.show()

## Section 4 — Trend Analysis

In [13]:
# ── Monthly revenue ───────────────────────────────────────────────────
monthly = (
    df.groupby("THANG_NAM")
    .agg(
        total           = ("THANHTIEN", "sum"),
        unique_visits   = ("ID_KHAMBENH", "nunique"),
        unique_patients = ("ID_BENHNHAN", "nunique"),
    )
    .reset_index()
    .sort_values("THANG_NAM")
)
monthly["growth_rate"] = monthly["total"].pct_change() * 100

# ── Daily series for STL ──────────────────────────────────────────────
daily = (
    df.groupby("NGAY")["THANHTIEN"].sum()
    .reset_index()
    .sort_values("NGAY")
    .set_index("NGAY")
)

trend_values = seasonal_values = residual_values = dates_list = []

if len(daily) >= 14:
    try:
        from statsmodels.tsa.seasonal import STL
        stl = STL(daily["THANHTIEN"], period=7, robust=True)
        result = stl.fit()
        trend_values   = [round(float(v), 0) for v in result.trend.tolist()]
        seasonal_values= [round(float(v), 0) for v in result.seasonal.tolist()]
        residual_values= [round(float(v), 0) for v in result.resid.tolist()]
        dates_list     = daily.index.strftime("%Y-%m-%d").tolist()
        print(f"✅ STL decomposition thành công ({len(dates_list)} ngày).")
    except Exception as e:
        print(f"⚠️  STL thất bại: {e}")
else:
    print(f"⚠️  Không đủ dữ liệu cho STL (cần ≥ 14 ngày, hiện có {len(daily)}).")

print("✅ Trend analysis đã tính xong.")

✅ STL decomposition thành công (130 ngày).
✅ Trend analysis đã tính xong.


In [14]:
# Doanh thu theo tháng + tỷ lệ tăng trưởng
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=monthly["THANG_NAM"], y=monthly["total"],
           name="Doanh thu", marker_color="#1976d2"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=monthly["THANG_NAM"], y=monthly["growth_rate"],
               name="Tăng trưởng (%)", mode="lines+markers",
               line=dict(color="#e53935", width=2)),
    secondary_y=True,
)
fig.update_layout(title="Doanh thu theo tháng & tỷ lệ tăng trưởng MoM")
fig.update_yaxes(title_text="Doanh thu (VNĐ)", secondary_y=False)
fig.update_yaxes(title_text="Tăng trưởng (%)",  secondary_y=True)
fig.show()

In [15]:
if dates_list:
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=("Trend (xu hướng dài hạn)", "Seasonal (mùa vụ 7 ngày)", "Residual (nhiễu)"),
        shared_xaxes=True,
    )
    fig.add_trace(go.Scatter(x=dates_list, y=trend_values,    name="Trend",    line=dict(color="#1976d2")), row=1, col=1)
    fig.add_trace(go.Scatter(x=dates_list, y=seasonal_values, name="Seasonal", line=dict(color="#26a69a")), row=2, col=1)
    fig.add_trace(go.Scatter(x=dates_list, y=residual_values, name="Residual", line=dict(color="#8e24aa", width=1)), row=3, col=1)
    fig.update_layout(title="STL Decomposition — Doanh thu theo ngày", height=600, showlegend=False)
    fig.show()
else:
    print("Không đủ dữ liệu để vẽ STL decomposition.")

In [16]:
# Doanh thu TB theo ngày trong tuần
fig = px.bar(
    dow_stats,
    x="THU_VI", y="mean",
    title="Doanh thu trung bình theo ngày trong tuần",
    labels={"THU_VI": "Ngày", "mean": "Doanh thu TB (VNĐ)"},
    color="mean",
    color_continuous_scale="Blues",
    text_auto=",.0f",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## Section 5 — Export JSON cho Dashboard

Chạy cell này để cập nhật dữ liệu phân tích lên dashboard.

In [17]:
generated_at = datetime.datetime.now().isoformat()

# ── descriptive_stats.json ────────────────────────────────────────────
descriptive_output = {
    "generated_at"      : generated_at,
    "overall"           : overall_stats,
    "bhyt_ratio"        : round(bhyt_ratio, 2),
    "revenue_per_patient": round(revenue_per_patient, 0),
    "revenue_per_visit" : round(revenue_per_visit, 0),
    "unique_patients"   : unique_patients,
    "unique_visits"     : unique_visits,
    "by_department"     : dept_stats.fillna(0).round(0).to_dict(orient="records"),
    "by_service_group"  : service_stats.fillna(0).round(0).to_dict(orient="records"),
    "by_day_of_week"    : dow_stats.fillna(0).to_dict(orient="records"),
}

out1 = OUTPUT_DIR / "descriptive_stats.json"
out1.write_text(json.dumps(descriptive_output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ {out1}")

# ── correlation_matrix.json ───────────────────────────────────────────
correlation_output = {
    "generated_at": generated_at,
    "department_correlation": {
        "labels": corr_dept.columns.tolist(),
        "matrix": corr_dept.round(3).values.tolist(),
    },
    "service_correlation": {
        "labels": corr_svc.columns.tolist(),
        "matrix": corr_svc.round(3).values.tolist(),
    },
    "dept_volume_revenue": dept_vol_rev.to_dict(orient="records"),
}

out2 = OUTPUT_DIR / "correlation_matrix.json"
out2.write_text(json.dumps(correlation_output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ {out2}")

# ── trend_analysis.json ───────────────────────────────────────────────
trend_output = {
    "generated_at"    : generated_at,
    "monthly_revenue" : monthly.fillna(0).round(0).to_dict(orient="records"),
    "seasonal_by_dow" : dow_stats.fillna(0).to_dict(orient="records"),
    "decomposition"   : {
        "dates"   : dates_list,
        "trend"   : trend_values,
        "seasonal": seasonal_values,
        "residual": residual_values,
    },
}

out3 = OUTPUT_DIR / "trend_analysis.json"
out3.write_text(json.dumps(trend_output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ {out3}")

print(f"\n🎉 Export hoàn tất lúc {generated_at[:19]}")
print("   → Dashboard tại http://127.0.0.1:5000/analysis")

✅ ../backend/analysis_output/descriptive_stats.json
✅ ../backend/analysis_output/correlation_matrix.json
✅ ../backend/analysis_output/trend_analysis.json

🎉 Export hoàn tất lúc 2026-05-24T02:53:10
   → Dashboard tại http://127.0.0.1:5000/analysis
